In [1]:
from io import StringIO
import json
import os

import boto3
import pandas as pd
pd.set_option("display.max_columns", 50)

In [2]:
aws_access_key = os.getenv("AWS_ACCESS_KEY")
aws_secret_key = os.getenv("AWS_SECRET_KEY")

In [3]:
s3 = boto3.client("s3", aws_access_key_id=aws_access_key, aws_secret_access_key=aws_secret_key)

In [4]:
bucket = "cubix-chicago-taxi-bb-v-2"
dim_community_areas_path = "transformed_data/dim_community_areas/dim_community_areas.csv"
dim_company_path = "transformed_data/dim_company/dim_company.csv"
dim_date_path = "transformed_data/dim_date/dim_date.csv"
dim_payment_type_path = "transformed_data/dim_payment_type/dim_payment_type.csv"
dim_weather_path = "transformed_data/dim_weather/"
fact_taxi_trips_path = "transformed_data/fact_taxi_trips/"


In [5]:
from io import StringIO
import json
from typing import Dict, List
import pandas as pd

def read_file_from_s3(
    s3, 
    bucket: str, 
    key: str, 
    file_format: str = "csv"
):

    """
    :param s3:          S3 client.
    :param bucket:      Name of the S3 Bucket where the file is stored.
    :param key:         Path within the S3 bucket.
    :param file_format: "json" or "csv".
    
    """
    response = s3.get_object(Bucket=bucket, Key=key)
    content = response["Body"].read().decode("utf-8")

    if file_format == "csv":
        return pd.read_csv(StringIO(content))
    if file_format == "json":
        return json.loads(content)
    else:
        raise ValueError("Unsupported file format, use 'csv' or 'json'.")

In [6]:
dim_community_areas = read_file_from_s3(s3, bucket, dim_community_areas_path)
dim_company         = read_file_from_s3(s3, bucket, dim_company_path)
dim_date            = read_file_from_s3(s3, bucket, dim_date_path)
dim_payment_type    = read_file_from_s3(s3, bucket, dim_payment_type_path)

In [7]:
print(dim_community_areas.head(1))
print(dim_company.head(1))
print(dim_date.head(1))
print(dim_payment_type.head(1))

   area_code community_are
0          1   Rogers Park
   company_id           company
0           1  Medallion Leasin
         date  year  months  day  day_of_week  is_weekend
0  2025-01-01  2025       1    1            3       False
   payment_type_id payment_type
0                1       Mobile


In [8]:
fact_taxi_trips_list = []
dim_weather_list = []

In [9]:
for file in s3.list_objects(Bucket=bucket, Prefix=fact_taxi_trips_path)["Contents"]:
    taxi_key = file["Key"]
    taxi_raw_file_name = file["Key"].split("/")[-1]

    if taxi_raw_file_name.split(".")[-1] == "csv":
        daily_file_name = fact_taxi_trips_path + taxi_raw_file_name
        taxi_trip_daily = read_file_from_s3(s3, bucket, daily_file_name)

        fact_taxi_trips_list.append(taxi_trip_daily)
        print(f"{taxi_raw_file_name} has been added")

fact_taxi_trips_2026-01-27.csv has been added
fact_taxi_trips_2026-01-28.csv has been added
fact_taxi_trips_2026-01-29.csv has been added
fact_taxi_trips_2026-01-30.csv has been added
fact_taxi_trips_2026-01-31.csv has been added
fact_taxi_trips_2026-02-01.csv has been added
fact_taxi_trips_2026-02-02.csv has been added
fact_taxi_trips_2026-02-03.csv has been added
fact_taxi_trips_2026-02-04.csv has been added
fact_taxi_trips_2026-02-05.csv has been added
fact_taxi_trips_2026-02-06.csv has been added
fact_taxi_trips_2026-02-07.csv has been added
fact_taxi_trips_2026-02-08.csv has been added
fact_taxi_trips_2026-02-09.csv has been added
fact_taxi_trips_2026-02-10.csv has been added
fact_taxi_trips_2026-02-11.csv has been added


In [10]:
fact_taxi_trips = pd.concat(fact_taxi_trips_list, ignore_index=True)

In [11]:
fact_taxi_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 242015 entries, 0 to 242014
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   trip_id                     242015 non-null  str    
 1   taxi_id                     242015 non-null  str    
 2   trip_start_timestamp        242015 non-null  str    
 3   trip_end_timestamp          242015 non-null  str    
 4   trip_seconds                242015 non-null  int64  
 5   trip_miles                  242015 non-null  float64
 6   pickup_community_area_id    242015 non-null  int64  
 7   dropoff_community_area_id   242015 non-null  int64  
 8   fare                        242015 non-null  float64
 9   tips                        242015 non-null  float64
 10  tolls                       242015 non-null  float64
 11  extras                      242015 non-null  float64
 12  trip_total                  242015 non-null  float64
 13  pickup_centroid_latitude 

In [12]:
for file in s3.list_objects(Bucket=bucket, Prefix=dim_weather_path)["Contents"]:
    weather_key = file["Key"]
    weather_raw_file_name = file["Key"].split("/")[-1]

    if weather_raw_file_name.split(".")[-1] == "csv":
        daily_file_name = dim_weather_path + weather_raw_file_name
        weather_daily = read_file_from_s3(s3, bucket, daily_file_name)

        dim_weather_list.append(weather_daily)
        print(f"{weather_raw_file_name} has been added")

weather_2026-01-27.csv has been added
weather_2026-01-28.csv has been added
weather_2026-01-29.csv has been added
weather_2026-01-30.csv has been added
weather_2026-01-31.csv has been added
weather_2026-02-01.csv has been added
weather_2026-02-02.csv has been added
weather_2026-02-03.csv has been added
weather_2026-02-04.csv has been added
weather_2026-02-05.csv has been added
weather_2026-02-06.csv has been added
weather_2026-02-07.csv has been added
weather_2026-02-08.csv has been added
weather_2026-02-09.csv has been added
weather_2026-02-10.csv has been added
weather_2026-02-11.csv has been added
weather_2026-02-12.csv has been added


In [13]:
dim_weather = pd.concat(dim_weather_list, ignore_index=True)


In [14]:
dim_weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 408 entries, 0 to 407
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   datetime       408 non-null    str    
 1   teperature     408 non-null    float64
 2   wind_speed     408 non-null    float64
 3   rain           408 non-null    float64
 4   percipitation  408 non-null    float64
dtypes: float64(4), str(1)
memory usage: 16.1 KB


#### Create datamodel

In [15]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips, dim_weather, left_on="datetime_for_weather", right_on="datetime")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=(["datetime", "datetime_for_weather"]))

fact_taxi_trips_full.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,payment_type_id,company_id,teperature,wind_speed,rain,percipitation
0,1402a73eb5409bd27226ff94796e254caef94018,2e82e26afb77e809fe4a44b02a152bdc079623600ae1b7...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,900,3.70,37,39,13.00,0.00,0.0,0.0,13.0,41.809084,-87.632425,41.808916,-87.596183,5,8,-11.9,15.2,0.0,0.0
1,3157b8da9dadbaf24734c2f8d07c148ef4802807,34f13b72cc0f5eee8a0827f77e3d51707f7cdf695e7b4a...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,1326,11.17,8,43,29.50,0.00,0.0,0.0,29.5,41.899602,-87.633308,41.761578,-87.572782,3,5,-11.9,15.2,0.0,0.0
2,3d7a524ca2142498775665a9a602c201ae3f5628,521928d8ec531b8101ca508e457ee15be5a46fe3ff429e...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,960,6.60,32,6,18.75,3.85,0.0,0.0,22.6,41.878866,-87.625192,41.944227,-87.655998,2,7,-11.9,15.2,0.0,0.0
3,44971fe06b6b0baa21f93cfeaadc52c03ed3b1c3,f35fcc7d28fd3ca0324ad6b42c6bf1dd0413320a9c4b65...,2026-01-27 23:45:00,2026-01-27T23:45:00.000,5,0.00,8,8,24.50,0.00,0.0,0.0,25.0,41.899602,-87.633308,41.899602,-87.633308,2,1,-11.9,15.2,0.0,0.0
4,50445c86bec724afea4a3b3d0892a3323a632faa,83a03e93bf25ce653700ec8360c79b5f393ab5c9167f31...,2026-01-27 23:45:00,2026-01-28T00:15:00.000,1380,17.30,76,28,43.00,9.50,0.0,4.0,56.5,41.980264,-87.913625,41.874005,-87.663518,2,8,-11.9,15.2,0.0,0.0


In [16]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_company, left_on="company_id", right_on="company_id")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=(["company_id"]))


In [17]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_payment_type, left_on="payment_type_id", right_on="payment_type_id")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=(["payment_type_id"]))

fact_taxi_trips_full.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,teperature,wind_speed,rain,percipitation,company,payment_type
0,1402a73eb5409bd27226ff94796e254caef94018,2e82e26afb77e809fe4a44b02a152bdc079623600ae1b7...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,900,3.70,37,39,13.00,0.00,0.0,0.0,13.0,41.809084,-87.632425,41.808916,-87.596183,-11.9,15.2,0.0,0.0,Transit Administrative Center Inc,Unknown
1,3157b8da9dadbaf24734c2f8d07c148ef4802807,34f13b72cc0f5eee8a0827f77e3d51707f7cdf695e7b4a...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,1326,11.17,8,43,29.50,0.00,0.0,0.0,29.5,41.899602,-87.633308,41.761578,-87.572782,-11.9,15.2,0.0,0.0,Flash Cab,Prcard
2,3d7a524ca2142498775665a9a602c201ae3f5628,521928d8ec531b8101ca508e457ee15be5a46fe3ff429e...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,960,6.60,32,6,18.75,3.85,0.0,0.0,22.6,41.878866,-87.625192,41.944227,-87.655998,-11.9,15.2,0.0,0.0,Chicago City Taxi Association,Credit Card
3,44971fe06b6b0baa21f93cfeaadc52c03ed3b1c3,f35fcc7d28fd3ca0324ad6b42c6bf1dd0413320a9c4b65...,2026-01-27 23:45:00,2026-01-27T23:45:00.000,5,0.00,8,8,24.50,0.00,0.0,0.0,25.0,41.899602,-87.633308,41.899602,-87.633308,-11.9,15.2,0.0,0.0,Medallion Leasin,Credit Card
4,50445c86bec724afea4a3b3d0892a3323a632faa,83a03e93bf25ce653700ec8360c79b5f393ab5c9167f31...,2026-01-27 23:45:00,2026-01-28T00:15:00.000,1380,17.30,76,28,43.00,9.50,0.0,4.0,56.5,41.980264,-87.913625,41.874005,-87.663518,-11.9,15.2,0.0,0.0,Transit Administrative Center Inc,Credit Card


In [18]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="pickup_community_area_id", right_on="area_code")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["pickup_community_area_id", "area_code"])
fact_taxi_trips_full.rename(columns={"community_area": "pickup_community_area_name"}, inplace=True)


In [19]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="dropoff_community_area_id", right_on="area_code")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["dropoff_community_area_id", "area_code"])
fact_taxi_trips_full.rename(columns={"community_area": "dropoff_community_area_name"}, inplace=True)


In [21]:
dim_date["date"] = pd.to_datetime(dim_date["date"])
fact_taxi_trips_full["trip_start_timestamp"] = pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"])

fact_taxi_trips_full["trip_start_date"] = pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"].dt.date)

In [22]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_date, left_on="trip_start_date", right_on="date")

fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["trip_start_date", "date"])

In [26]:
fact_taxi_trips_full = fact_taxi_trips_full.rename(columns={
    "community_are_x": "pickup_community_area_name",
    "community_are_y": "dropoff_community_area_name"
})

In [27]:
fact_taxi_trips_full.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,teperature,wind_speed,rain,percipitation,company,payment_type,pickup_community_area_name,dropoff_community_area_name,year,months,day,day_of_week,is_weekend
0,1402a73eb5409bd27226ff94796e254caef94018,2e82e26afb77e809fe4a44b02a152bdc079623600ae1b7...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,900,3.70,13.00,0.00,0.0,0.0,13.0,41.809084,-87.632425,41.808916,-87.596183,-11.9,15.2,0.0,0.0,Transit Administrative Center Inc,Unknown,Fuller Park,Kenwood,2026,1,27,2,False
1,3157b8da9dadbaf24734c2f8d07c148ef4802807,34f13b72cc0f5eee8a0827f77e3d51707f7cdf695e7b4a...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,1326,11.17,29.50,0.00,0.0,0.0,29.5,41.899602,-87.633308,41.761578,-87.572782,-11.9,15.2,0.0,0.0,Flash Cab,Prcard,Near North Side,South Shore,2026,1,27,2,False
2,3d7a524ca2142498775665a9a602c201ae3f5628,521928d8ec531b8101ca508e457ee15be5a46fe3ff429e...,2026-01-27 23:45:00,2026-01-28T00:00:00.000,960,6.60,18.75,3.85,0.0,0.0,22.6,41.878866,-87.625192,41.944227,-87.655998,-11.9,15.2,0.0,0.0,Chicago City Taxi Association,Credit Card,Loop,Lake View,2026,1,27,2,False
3,44971fe06b6b0baa21f93cfeaadc52c03ed3b1c3,f35fcc7d28fd3ca0324ad6b42c6bf1dd0413320a9c4b65...,2026-01-27 23:45:00,2026-01-27T23:45:00.000,5,0.00,24.50,0.00,0.0,0.0,25.0,41.899602,-87.633308,41.899602,-87.633308,-11.9,15.2,0.0,0.0,Medallion Leasin,Credit Card,Near North Side,Near North Side,2026,1,27,2,False
4,50445c86bec724afea4a3b3d0892a3323a632faa,83a03e93bf25ce653700ec8360c79b5f393ab5c9167f31...,2026-01-27 23:45:00,2026-01-28T00:15:00.000,1380,17.30,43.00,9.50,0.0,4.0,56.5,41.980264,-87.913625,41.874005,-87.663518,-11.9,15.2,0.0,0.0,Transit Administrative Center Inc,Credit Card,O'Hare[11],Near West Side,2026,1,27,2,False
